In [1]:
'''
Projeto de IA (N2) - Recomendação musical por agregação de vizinhança aprendida
Integrantes: Arthur Jones Bicalho dos Santos 10420317;
             Luis Felipe Santos do Nascimento 10420572
Síntese: grafo de caminhadas aleatórias e grade de hiperparâmetros do GraphSAGE na validação.
Histórico:
2026-11-10 | Arthur | Criação da célula de sensibilidade (no notebook 02)
2026-11-13 | Luis Felipe | Notebook próprio (o kernel do 02 era encerrado por falta de memória), grade em processos paralelos com cache
2026-11-18 | Luis Felipe | Atualização da lista de integrantes
'''

'\nProjeto de IA (N2) - Recomendação musical por agregação de vizinhança aprendida\nIntegrantes: Arthur Jones Bicalho dos Santos 10420317;\n             Luis Felipe Santos do Nascimento 10420572\nSíntese: grafo de caminhadas aleatórias e grade de hiperparâmetros do GraphSAGE na validação.\nHistórico:\n2026-11-10 | Arthur | Criação da célula de sensibilidade (no notebook 02)\n2026-11-13 | Luis Felipe | Notebook próprio (o kernel do 02 era encerrado por falta de memória), grade em processos paralelos com cache\n'

In [2]:
# Localiza a raiz do repositório (pasta que contém src/) e passa a trabalhar a partir dela.
# Sem isto, abrir o notebook dentro de notebooks/ quebra os imports de src e os caminhos
# relativos data/ e results/ (FileNotFoundError).
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data.py').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.makedirs('results', exist_ok=True)
print('Raiz do projeto:', ROOT)

Raiz do projeto: projeto_ia


## Grafo de caminhadas e sensibilidade a hiperparâmetros do GraphSAGE (validação)
O treino é feito em CPU (cerca de 40 s por época na configuração maior), então a grade roda em processos paralelos e com orçamento de `GRID_EPOCHS` épocas por configuração. Um teste preliminar com `lr=1e-3` chegou a NDCG@10 de 0,013 em 5 épocas (abaixo da popularidade), por isso a grade usa taxas de aprendizado maiores.

In [3]:
import pandas as pd
from src.data import load_processed
D = load_processed()

import time
from src.walks import *
from src.experiments import grid_job, run_parallel

t0 = time.perf_counter()
indptr, indices = build_adj(D['train'], D['nU'], D['nA'])
C = random_walk_counts(indptr, indices, num_walks=50, walk_len=6, seed=0)
ei, ew = walk_graph(C, 20)
print(f'Caminhadas e vizinhança top-20: {time.perf_counter() - t0:.1f} s | nós: {C.shape[0]} | arestas: {ei.shape[1]}')

GRID_EPOCHS = 30
jobs = [dict(T=T, d=d, lr=lr, max_epochs=GRID_EPOCHS, threads=2)
        for T in (10, 20) for d in (32, 64) for lr in (3e-3, 1e-2)]
grade = pd.DataFrame(run_parallel(grid_job, jobs, workers=4)).sort_values('val_ndcg10', ascending=False)
grade.to_csv('results/sensibilidade_hiperparametros.csv', index=False)
grade

Caminhadas e vizinhança top-20: 2.0 s | nós: 31455 | arestas: 629100


,T,d,lr,val_ndcg10,melhor_epoca,epocas,segundos
5,20,32,0.010,0.080150,30,30,1673
7,20,64,0.010,0.075116,30,30,2044
1,10,32,0.010,0.071113,30,30,941
6,20,64,0.003,0.062811,30,30,2049
3,10,64,0.010,0.062460,30,30,1568
4,20,32,0.003,0.055992,30,30,1680
2,10,64,0.003,0.053980,30,30,1566
0,10,32,0.003,0.050801,30,30,939
